# AutoDock Tutorial
### Notebook 01. Ligand Preparation for Docking  

**Version 1.0.0 - April, 2026. Monterrey**


**Authors:** 
[Ana C. Murrieta ](https://orcid.org/0000-0002-7619-8880) and [Flavio F. Contreras-Torres](https://orcid.org/0000-0003-2375-131X). Tecnológico de Monterrey.



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/AutodockTutorial/blob/main/notebooks/pdb_structure_preparation.ipynb)

---


## Contents


This notebook begins with the definition of the computational environment required for sequence hand


The activities are structured to be completed in approximately **60 to 120 minutes**. However, this is your personal workspace—we encourage you to add your own code cells, inspect additional PDB entries, and test alternative structural decisions as you build confidence with the workflow. The more you explore, the better prepared you'll be for the receptor preparation and docking steps that follow.


---

## How to work
1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourID_ligand_preparation_docking.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

### Theoretical Background

## Open Babel 

[Open Babel](https://openbabel.org/) is a widely used open-source chemical toolbox that provides a comprehensive suite of tools for converting, manipulating, and analyzing chemical data. It supports over 100 chemical file formats, making it a versatile choice for tasks like ligand preparation for docking. Open Babel excels at rapid format conversion and basic structure manipulation, but it may not always capture complex chemical features with the same level of accuracy as more specialized tools. For example, while it can convert a ligand from SDF to PDBQT format, it may not always preserve detailed stereochemistry or handle macrocycles as effectively as other software. However, its speed and broad format support make it a popular choice for high-throughput virtual screening workflows where rapid processing is essential.



## Meeko: Molecule Parametrization and Software Interoperability

[Meeko](https://github.com/forlilab/Meeko) is a Python package developed for the parametrization of ligands and receptors within the AutoDock suite. It utilizes the **RDKit** library for chemical perception, which allows it to define molecular properties such as connectivity, bond orders, and formal charges independently of atom-naming conventions.

### Technical Characteristics:  
* **Scope of Functionality:** Meeko is a **parametrization tool** and does not include functions for **energy minimization** or geometric optimization. Molecules must be prepared with 3D coordinates prior to processing.
* **Protonation Handling:** The software perceives and preserves the **protonation state** existing in the input file. It does not dynamically calculate $pK_a$ values to add or remove protons; instead, it relies on explicit hydrogens provided by the user or external tools to match its residue templates.
* **Specialized Methods:** It supports advanced docking protocols, such as **macrocycle flexibility**, **explicitly hydrated docking**, and **reactive docking**.
* **Computational Performance:** In benchmarks involving 10,000 molecules, Meeko completed processing in 37.3 seconds, compared to 8.4 seconds for Open Babel and 2,200 seconds for MGLTools.
* **File Formats and Interoperability:** The package facilitates the use of the **SDF** format for docking inputs and outputs. Additionally, it allows for the conversion of docking results back into RDKit molecules, enabling integration with Molecular Dynamics (MD) and other computational chemistry pipelines.



### Step 0. Computational Environment and Required Tools

Before starting receptor and ligand preparation, verify that the working environment includes the software required for structure inspection, protonation, visualization, ligand refinement, and docking file generation.

This notebook relies on the following **Python libraries**:
- **Biopython** – for PDB/PQR structure parsing and manipulation (Steps 1–3, 5A)
- **RDKit** – for ligand validation and inspection (Step 4A)
- **pathlib**, **shutil**, **subprocess** – standard library modules for file handling and external command execution
- **PyYAML** – for loading atom-type rules in Step 5A

The following **external command-line tools** are required and must be available in the active environment:

| Tool | Purpose | Step(s) | Verification Command |
|------|---------|---------|---------------------|
| **PDB2PQR** | Receptor protonation at pH 7.4 | Step 3 | `pdb2pqr30` or `pdb2pqr` |
| **Open Babel** | Ligand hydrogenation, minimization, and PDBQT conversion | Steps 4A, 4B, 5B, 5C1, 5C2, 5C3 | `obabel` and `obminimize` |
| **Meeko** | Receptor PDBQT generation and alternative ligand PDBQT parametrization | Step 5A, 5D | `mk_prepare_receptor.py` or `mk_prepare_ligand.py` |
| **PyMOL** (optional) | Visual inspection of structures | All steps | `pymol` |


Additionally, Step 5A requires an **atom rules configuration file** (`atom_rules.yaml`), which should be included in the tutorial repository.

This file defines protein atom names and two-letter element mappings used during PQR → PDB conversion.

> **Note:** This notebook assumes that all required tools are already installed and accessible from the active environment. If any tool is missing, the corresponding step will fail with a descriptive error message.

You can verify the availability of each tool by running the following checks:

```python
import shutil

# PDB2PQR
print("PDB2PQR:", shutil.which("pdb2pqr30") or shutil.which("pdb2pqr"))

# Open Babel
print("obabel:", shutil.which("obabel") or shutil.which("babel"))
print("obminimize:", shutil.which("obminimize"))

# Meeko
print("Meeko:", 
      shutil.which("mk_prepare_receptor.py") or 
      shutil.which("mk_prepare_receptor") or 
      shutil.which("prepare_receptor"))

# Optional: PyMOL
print("PyMOL:", shutil.which("pymol"))
```

If any result is `None`, the corresponding tool is not available in the active environment and should be installed before proceeding.

## Step 4A. Inspect and hydrogenate the reference ligand

After defining the chemical state of the receptor, the next step is to prepare the crystallographic ligand that will be used as a structural reference for docking. At this stage, the ligand is loaded and inspected to confirm that its geometry, connectivity, and 3D coordinates are chemically reasonable.

Explicit hydrogens are then added to the ligand so that its structure can be evaluated in a chemically complete form before further refinement. The hydrogenated ligand should be inspected visually to confirm that atom connectivity and hydrogen placement are consistent with the expected molecular structure.

In practical terms, this step generates an intermediate ligand structure that is chemically more complete and ready for geometry refinement in the following minimization step.

In [ ]:
# Step 4A. Inspect and hydrogenate the reference ligand
# Use Open Babel to add hydrogens at near-physiological pH and generate
# an intermediate ligand PDB for visual inspection before minimization.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "prepared").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\nCould not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

# -------------------------------------------------------------------
# Locate ligand file
# -------------------------------------------------------------------
prepared_dir = project_root / "docking" / "prepared"
ligand_input_path = prepared_dir / "5YCP_BRL_ligand.pdb"

if not ligand_input_path.exists():
    raise FileNotFoundError(
        f"\nLigand file not found:\n   {ligand_input_path}\n\n"
        "   Expected crystallographic ligand file: 5YCP_BRL_ligand.pdb"
    )

print("Using ligand:")
print(f"  File: {ligand_input_path.name}")
print(f"  Path: {ligand_input_path.resolve()}")

# -------------------------------------------------------------------
# Locate Open Babel executable
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\nOpen Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' is available from the command line."
    )

# -------------------------------------------------------------------
# Define output file
# -------------------------------------------------------------------
target_ph = 7.4
ligand_h_path = prepared_dir / "5YCP_BRL_ligand_withH.pdb"

# -------------------------------------------------------------------
# Hydrogenate ligand with Open Babel
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    str(ligand_input_path),
    "-O", str(ligand_h_path),
    "-p", str(target_ph),
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\nOpen Babel timed out after 300 seconds.\n"
        "   Check the ligand file or environment configuration."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\nOpen Babel failed during ligand hydrogenation.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_h_path.exists():
    raise FileNotFoundError(
        f"\nHydrogenated ligand file was not created:\n   {ligand_h_path}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDB
# -------------------------------------------------------------------
atom_count = 0
hetatm_count = 0
hydrogen_like_count = 0

with open(ligand_h_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith("ATOM"):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

        elif line.startswith("HETATM"):
            hetatm_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

total_coordinate_records = atom_count + hetatm_count

if total_coordinate_records == 0:
    raise ValueError(
        "\nThe hydrogenated ligand PDB contains no ATOM/HETATM records.\n"
        "   Inspect the Open Babel output file."
    )

if hydrogen_like_count == 0:
    print("\nWarning: No hydrogen atoms were detected in the output PDB.")
    print("   Open Babel may not have added hydrogens as expected.")
    print("   Inspect the file manually before proceeding.")

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nLigand hydrogenation summary")
print("-" * 60)
print(f"Input ligand:             {ligand_input_path.name}")
print(f"Hydrogenated output:      {ligand_h_path.name}")
print(f"Target pH:                {target_ph}")
print(f"ATOM records:             {atom_count}")
print(f"HETATM records:           {hetatm_count}")
print(f"Total coordinate records: {total_coordinate_records}")
print(f"Hydrogen-like atoms:      {hydrogen_like_count}")

print("\nVisual inspection is recommended before minimization.")
print("   Open the output PDB in PyMOL and confirm that:")
print("   - the ligand connectivity is reasonable")
print("   - hydrogens were added correctly")
print("   - no abnormal geometry was introduced")

# Optional: update working variable for downstream steps
ligand_hydrogenated_path = ligand_h_path

> **Output note:**  
> In this step, the crystallographic ligand is hydrogenated under near-physiological conditions to generate an intermediate structure suitable for visual inspection before energy minimization. This confirms that the ligand can be processed successfully with **Open Babel** and that a hydrogenated working file can be produced for the next stage of preparation.  
>
> In this example, the input ligand `5YCP_BRL_ligand.pdb` was processed at **pH 7.4**, generating the hydrogenated output file `5YCP_BRL_ligand_withH.pdb`. The resulting structure contained **44 coordinate records** in total, including **19 hydrogen-like atoms**, indicating that hydrogen addition was successfully applied.  
>
> At this stage, the hydrogenated ligand should be inspected visually in **PyMOL** to confirm that the connectivity remains reasonable, hydrogens were added in plausible positions, and no abnormal geometry was introduced during preparation.  
>
> In practical terms, this step yields a hydrogenated ligand structure ready for visual validation and subsequent geometry minimization prior to docking-file conversion.

## Step 4B. Minimize the ligand geometry

Once the hydrogenated ligand has been inspected and no obvious connectivity or geometry problems are observed, the next step is to refine its structure by energy minimization. This helps relieve unfavorable contacts introduced during hydrogen addition and produces a more relaxed ligand conformation for downstream docking preparation.

In this workflow, minimization is carried out with **Open Babel** using the `obminimize` utility and the **MMFF94** force field. The purpose of this step is not to generate a completely new ligand conformation, but to obtain a geometrically improved version of the hydrogenated structure while preserving its overall chemical identity.

The output of this step is a **minimized ligand PDB file** that can be inspected again in **PyMOL** before conversion to the docking format required by the selected engine. For typical organic ligands, **MMFF94** is a reasonable choice, although ligands containing metals or unsupported atom types may require a different force field or preparation workflow.

In [ ]:
# Step 4B. Minimize the ligand geometry
# Use Open Babel's obminimize to relax the hydrogenated ligand geometry
# with the MMFF94 force field before docking-file conversion.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "prepared").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\nCould not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

prepared_dir = project_root / "docking" / "prepared"

# -------------------------------------------------------------------
# Resolve input ligand from Step 4A or from disk
# -------------------------------------------------------------------
if "ligand_hydrogenated_path" in globals():
    ligand_input_path = Path(ligand_hydrogenated_path)
else:
    ligand_input_path = prepared_dir / "5YCP_BRL_ligand_withH.pdb"

if not ligand_input_path.exists():
    raise FileNotFoundError(
        f"\nHydrogenated ligand file not found:\n   {ligand_input_path}\n\n"
        "   Please run Step 4A first, or make sure '5YCP_BRL_ligand_withH.pdb' exists."
    )

print("Using hydrogenated ligand:")
print(f"  File: {ligand_input_path.name}")
print(f"  Path: {ligand_input_path.resolve()}")

# -------------------------------------------------------------------
# Locate obminimize executable
# -------------------------------------------------------------------
obminimize_exe = shutil.which("obminimize")

if obminimize_exe is None:
    raise EnvironmentError(
        "\nobminimize was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obminimize' is available from the command line."
    )

# -------------------------------------------------------------------
# Define output file and minimization settings
# -------------------------------------------------------------------
ligand_min_path = prepared_dir / "5YCP_BRL_ligand_min.pdb"
forcefield = "MMFF94"
max_steps = 20000
convergence = "1e-6"

cmd = [
    obminimize_exe,
    "-ff", forcefield,
    "-n", str(max_steps),
    "-c", convergence,
    str(ligand_input_path),
]

print("\nRunning obminimize...")
print("Command:")
print(" ".join(cmd))
print(f"Output file: {ligand_min_path.name}")

# -------------------------------------------------------------------
# Run minimization and write stdout to output PDB
# -------------------------------------------------------------------
try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\nobminimize timed out after 600 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\nobminimize failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

with open(ligand_min_path, "w", encoding="utf-8") as fh:
    fh.write(result.stdout)

if not ligand_min_path.exists():
    raise FileNotFoundError(
        f"\nMinimized ligand file was not created:\n   {ligand_min_path}"
    )

# -------------------------------------------------------------------
# Basic validation of minimized output
# -------------------------------------------------------------------
atom_count = 0
hetatm_count = 0
hydrogen_like_count = 0

with open(ligand_min_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith("ATOM"):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

        elif line.startswith("HETATM"):
            hetatm_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

total_coordinate_records = atom_count + hetatm_count

if total_coordinate_records == 0:
    raise ValueError(
        "\nThe minimized ligand file contains no ATOM/HETATM records.\n"
        "   Inspect the obminimize output manually."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nLigand minimization summary")
print("-" * 60)
print(f"Input ligand:             {ligand_input_path.name}")
print(f"Minimized output:         {ligand_min_path.name}")
print(f"Force field:              {forcefield}")
print(f"Maximum steps:            {max_steps}")
print(f"Convergence criterion:    {convergence}")
print(f"ATOM records:             {atom_count}")
print(f"HETATM records:           {hetatm_count}")
print(f"Total coordinate records: {total_coordinate_records}")
print(f"Hydrogen-like atoms:      {hydrogen_like_count}")

print("\nNote: MMFF94 is appropriate for typical organic ligands.")
print("   Ligands containing metals or unsupported elements may require")
print("   a different force field or a different preparation workflow.")

print("\nVisual inspection is recommended after minimization.")
print("   Compare the minimized ligand against the hydrogenated input in PyMOL")
print("   to confirm that the geometry was relaxed without introducing distortions.")

# Optional: update working variable for downstream steps
ligand_minimized_path = ligand_min_path

> **Output note:**  
> In this step, the hydrogenated ligand is refined by energy minimization in order to relax its geometry before docking-file preparation. This confirms that the ligand can be processed successfully with **Open Babel** using `obminimize` and that the selected force field is compatible with the current ligand structure.  
>
> In this example, the hydrogenated input file `5YCP_BRL_ligand_withH.pdb` was minimized with the **MMFF94** force field using a maximum of **20,000 steps** and a convergence criterion of **1e-6**. The procedure completed successfully and generated the minimized ligand file `5YCP_BRL_ligand_min.pdb`.  
>
> The resulting minimized structure retained **44 coordinate records** in total, including **19 hydrogen-like atoms**, indicating that the ligand was minimized without loss of atoms during the process.  
>
> For typical organic ligands, **MMFF94** is a reasonable force field for this stage, although ligands containing metals or unsupported atom types may require a different preparation strategy.  
>
> In practical terms, this step yields a hydrogenated and geometrically relaxed ligand structure that can now be inspected visually and carried forward to docking-format conversion.

## Step 5B. Convert the reference ligand to docking format

In this step, the minimized **reference ligand** is converted into the docking format required by the selected engine. For AutoDock Vina workflows, this means generating a **PDBQT** file from the ligand structure prepared in **Step 4B**.

The starting point is the minimized ligand in **PDB** format. This structure is passed to **Open Babel** to generate a **PDBQT** file while preserving the ligand identity, retaining all hydrogens, and assigning AutoDock-compatible atom types during the conversion.

At this stage, the objective is not to alter the ligand geometry further, but to produce a docking-compatible ligand representation that can be used together with the prepared receptor. A basic validation is also performed to confirm that the output file was created successfully and contains the expected atomic records and torsion tree information.

In practical terms, this step generates the final **PDBQT** file for the crystallographic or reference ligand, which can be used for docking setup, protocol validation, or comparison with external test ligands prepared later in the workflow.

In [ ]:
# Step 5B. Convert the ligand to docking format
# Convert the minimized ligand PDB into PDBQT for AutoDock Vina,
# preserving hydrogens and atom identity information as much as possible.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "prepared").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\n Could not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

prepared_dir = project_root / "docking" / "prepared"

# -------------------------------------------------------------------
# Resolve minimized ligand from Step 4B or from disk
# -------------------------------------------------------------------
if "ligand_min_path" in globals():
    input_ligand = Path(ligand_min_path)
else:
    candidates = sorted(prepared_dir.glob("*_min.pdb"))
    if not candidates:
        raise FileNotFoundError(
            f"\n No minimized ligand file was found in:\n   {prepared_dir}\n\n"
            "   Please run Step 4B first."
        )
    input_ligand = candidates[-1]

if not input_ligand.exists():
    raise FileNotFoundError(
        f"\n Minimized ligand file not found:\n   {input_ligand}"
    )

print("Using minimized ligand:")
print(f"  File: {input_ligand.name}")
print(f"  Path: {input_ligand.resolve()}")

# -------------------------------------------------------------------
# Detect Open Babel
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\n Open Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' or 'babel'\n"
        "   is available from the command line."
    )

# -------------------------------------------------------------------
# Define output file
# -------------------------------------------------------------------
ligand_pdbqt_path = input_ligand.parent / f"{input_ligand.stem}.pdbqt"

if ligand_pdbqt_path.exists():
    ligand_pdbqt_path.unlink()

# -------------------------------------------------------------------
# Build Open Babel command
# -xh : preserve hydrogens
# -xn : preserve atom names
# -xp : preserve atom indices
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    str(input_ligand),
    "-O", str(ligand_pdbqt_path),
    "-xh",
    "-xn",
    "-xp",
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Open Babel timed out after 300 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand conversion to PDBQT failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdbqt_path.exists():
    raise FileNotFoundError(
        f"\n Ligand PDBQT file was not created:\n   {ligand_pdbqt_path}"
    )

# -------------------------------------------------------------------
# Count atoms in input PDB
# -------------------------------------------------------------------
input_atom_count = 0
with open(input_ligand, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            input_atom_count += 1

if input_atom_count == 0:
    raise ValueError(
        "\n The minimized ligand PDB contains no ATOM/HETATM records.\n"
        "   Inspect the input ligand before proceeding."
    )

# -------------------------------------------------------------------
# Basic validation of output PDBQT
# -------------------------------------------------------------------
output_atom_count = 0
tree_records = 0
remark_lines = 0

with open(ligand_pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            output_atom_count += 1
        elif line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            tree_records += 1
        elif line.startswith("REMARK"):
            remark_lines += 1

if output_atom_count == 0:
    raise ValueError(
        "\n The ligand PDBQT file contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nLigand docking-format conversion summary")
print("-" * 60)
print(f"Input ligand:             {input_ligand.name}")
print(f"Output ligand PDBQT:      {ligand_pdbqt_path.name}")
print(f"Full path:                {ligand_pdbqt_path.resolve()}")
print(f"Input ATOM/HETATM:        {input_atom_count}")
print(f"Output ATOM/HETATM:       {output_atom_count}")
print(f"Tree-related records:     {tree_records}")
print(f"REMARK lines:             {remark_lines}")

if output_atom_count != input_atom_count:
    print("\n Warning: Atom count changed during conversion.")
    print(f"   Input atoms:  {input_atom_count}")
    print(f"   Output atoms: {output_atom_count}")
    print("   Inspect the ligand carefully before docking.")
else:
    print("\n Atom count was preserved during conversion.")

print("\n Note:")
print("   This PDBQT conversion step is intended for AutoDock Vina workflows.")
print("   If you plan to use AutoDock4, ligand charge handling may require")
print("   a different preparation route.")

print("\n Visual inspection is recommended.")
print("   Open the ligand PDBQT in PyMOL or a text editor and confirm that:")
print("   - atom records are present")
print("   - hydrogens were preserved as expected")
print("   - the file is not empty or truncated")
print("   - the ligand identity is preserved after conversion")

# Optional: update working variable for downstream steps
ligand_pdbqt_path = ligand_pdbqt_path

> **Output note:**  
> In this step, the minimized reference ligand was successfully converted from **PDB** to **PDBQT**, generating the docking-ready ligand file required for AutoDock Vina. The conversion was performed with **Open Babel**, which automatically preserves hydrogens, atom names, and atom indices during PDBQT generation.  
>
> In this example, the input file `5YCP_BRL_ligand_min.pdb` was converted into `5YCP_BRL_ligand_min.pdbqt` and saved in the `docking/prepared/` directory. The resulting **PDBQT** file contained **44 ATOM/HETATM records**, which is the expected atom count for the fully hydrogenated BRL ligand (rosiglitazone, C₁₈H₁₉N₃O₃S). This indicates that atom preservation during conversion was successful and that the ligand identity was maintained consistently in the docking-ready file.  
>
> Additional records related to the ligand tree representation (`ROOT`, `BRANCH`, `ENDBRANCH`, `TORSDOF`) were also written, which is expected in the **PDBQT** format for ligand flexibility handling. In practical terms, this step confirms that the reference ligand is now available in its final docking-compatible representation and can be used for docking setup, protocol validation, or comparison with external ligands prepared later in the workflow.

## Step 5C. Prepare an External Ligand from SDF for Docking

For ligands other than the co-crystallized ligand present in the receptor structure, the preparation workflow accepts an external **SDF** file containing three-dimensional coordinates. Such files are commonly available from chemical databases such as **PubChem** and **ZINC**.

The preparation proceeds in three sequential steps:

1. **Step 5C1 – Protonation (SDF → PDB)**  
   The SDF file is converted to **PDB** format using **Open Babel**. Hydrogens are added and protonation states are assigned at **pH 7.4** using the `-p 7.4` option. The output is a PDB file containing the ligand with explicit hydrogens and a protonation state consistent with physiologically reasonable conditions.

2. **Step 5C2 – Energy Minimization (PDB → Minimized PDB)**  
   The protonated PDB structure is subjected to energy minimization using the **MMFF94** force field as implemented in Open Babel’s `obminimize` utility. A maximum of **20,000 steps** is specified together with a convergence criterion of **1 × 10⁻⁶**. This step relaxes bond lengths, angles, and torsions toward a local energy minimum before docking.

3. **Step 5C3 – Docking Format Conversion (Minimized PDB → PDBQT)**  
   The minimized PDB structure is converted to **PDBQT** format using **Open Babel**. In this format, non-polar hydrogens bonded to carbon are typically merged into the parent heavy atom, whereas polar hydrogens bonded to heteroatoms are retained explicitly. The output also includes the atom-type assignments and torsion-tree definitions required by **AutoDock Vina**.

Validation of the final **PDBQT** file includes verification of atom-record count, presence of torsion-tree records, and confirmation that the total partial charge is chemically reasonable for the expected ligand charge state.

Together, these three steps produce a protonated, minimized, and docking-ready ligand structure in **PDBQT** format suitable for molecular docking with **AutoDock Vina**.

### Input Requirements

| Requirement | Description |
|-------------|-------------|
| **File format** | SDF (Structure Data File) |
| **File location** | `docking/sdf/` |
| **3D coordinates** | Required (Open Babel does not generate 3D from 2D) |
| **Protonation state** | Not required in input (Step 5C1 adds hydrogens at pH 7.4) |


### How to Specify the Input Ligand

You can provide the input ligand in one of two ways:

1. **Automatic detection** (recommended for a single file):  
   Place exactly one `.sdf` file in the `docking/sdf/` directory. The notebook will automatically detect and use it.

2. **Explicit specification** (required for multiple files):  
   If multiple `.sdf` files are present, define the variable before running this step:
   
   ```python
   user_ligand_sdf = "CID_10198397.sdf"
   ```

In [ ]:
# Step 5C1. Convert an external ligand from SDF to PDB with hydrogens

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# User-configurable variable
# Example:
# user_ligand_sdf = "CID_10198397.sdf"
# -------------------------------------------------------------------
user_ligand_sdf = None

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "sdf").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\n Could not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

sdf_dir = project_root / "docking" / "sdf"

if not sdf_dir.exists():
    raise FileNotFoundError(
        f"\n Required folder not found:\n   {sdf_dir}\n\n"
        "   Create the folder 'docking/sdf/' and place your ligand .sdf file there."
    )

# -------------------------------------------------------------------
# Select input SDF
# -------------------------------------------------------------------
if user_ligand_sdf is not None:
    user_ligand_sdf_path = Path(user_ligand_sdf)
    input_sdf = user_ligand_sdf_path if user_ligand_sdf_path.is_absolute() else sdf_dir / user_ligand_sdf_path.name
else:
    sdf_candidates = sorted(sdf_dir.glob("*.sdf"))

    if len(sdf_candidates) == 0:
        raise FileNotFoundError(
            f"\n No SDF ligand file was found in:\n   {sdf_dir}\n\n"
            "   Place your ligand .sdf file there, or set:\n"
            "   user_ligand_sdf = 'your_file.sdf'"
        )
    elif len(sdf_candidates) > 1:
        raise ValueError(
            "\n Multiple SDF files were found in docking/sdf/.\n"
            "   Please set the variable:\n"
            "   user_ligand_sdf = 'your_file.sdf'\n\n"
            f"   Detected files:\n   " + "\n   ".join(p.name for p in sdf_candidates)
        )
    else:
        input_sdf = sdf_candidates[0]

if not input_sdf.exists():
    raise FileNotFoundError(f"\n Input SDF file not found:\n   {input_sdf}")

if input_sdf.suffix.lower() != ".sdf":
    raise ValueError(f"\n Expected an SDF file, but received:\n   {input_sdf.name}")

print("Using external ligand:")
print(f"  File: {input_sdf.name}")
print(f"  Path: {input_sdf.resolve()}")

# -------------------------------------------------------------------
# Detect Open Babel
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\n Open Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' or 'babel' is available."
    )

# -------------------------------------------------------------------
# Define output PDB
# -------------------------------------------------------------------
ligand_pdb_path = input_sdf.with_suffix(".pdb")

if ligand_pdb_path.exists():
    ligand_pdb_path.unlink()

# -------------------------------------------------------------------
# Build command: SDF -> PDB with hydrogens
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    str(input_sdf),
    "-O", str(ligand_pdb_path),
    "-p", "7.4",
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n SDF-to-PDB conversion timed out after 300 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n SDF-to-PDB conversion failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdb_path.exists():
    raise FileNotFoundError(
        f"\n Output PDB file was not created:\n   {ligand_pdb_path}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDB
# -------------------------------------------------------------------
atom_count = 0
hydrogen_count = 0

with open(ligand_pdb_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element in ("H", "D"):
                hydrogen_count += 1

if atom_count == 0:
    raise ValueError(
        "\n The generated PDB contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

print("\nExternal ligand conversion summary")
print("-" * 60)
print(f"Input SDF ligand:          {input_sdf.name}")
print(f"Output PDB ligand:         {ligand_pdb_path.name}")
print(f"Full path:                 {ligand_pdb_path.resolve()}")
print(f"ATOM/HETATM records:       {atom_count}")
print(f"Hydrogen-like atoms:       {hydrogen_count}")

print("\n Visual inspection is recommended before minimization.")
print("   Open the output PDB in PyMOL and confirm that:")
print("   - the ligand connectivity is reasonable")
print("   - hydrogens are present")
print("   - no abnormal geometry was introduced")

print("\n Note:")
print("   This workflow assumes that the input ligand is a 3D SDF file.")
print("   If the SDF lacks reliable 3D coordinates, inspect the structure")
print("   carefully before minimization and docking.")


# Optional variable for downstream steps
external_ligand_pdb_path = ligand_pdb_path

---

In [ ]:
# Step 5C2. Minimize the external ligand PDB

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Input from Step 5C1
# -------------------------------------------------------------------
if "external_ligand_pdb_path" not in globals():
    raise NameError(
        "Variable 'external_ligand_pdb_path' was not found. Please run Step 5C1 first."
    )

input_ligand = Path(external_ligand_pdb_path)

if not input_ligand.exists():
    raise FileNotFoundError(
        f"\n Input ligand PDB file not found:\n   {input_ligand}"
    )

print("Using external ligand PDB:")
print(f"  File: {input_ligand.name}")
print(f"  Path: {input_ligand.resolve()}")

# -------------------------------------------------------------------
# Detect obminimize
# -------------------------------------------------------------------
obminimize_exe = shutil.which("obminimize")

if obminimize_exe is None:
    raise EnvironmentError(
        "\n obminimize was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obminimize' is available."
    )

# -------------------------------------------------------------------
# Output file
# -------------------------------------------------------------------
ligand_min_path = input_ligand.with_name(f"{input_ligand.stem}_min.pdb")

if ligand_min_path.exists():
    ligand_min_path.unlink()

# -------------------------------------------------------------------
# Minimization settings
# -------------------------------------------------------------------
force_field = "MMFF94"
max_steps = 20000
convergence = "1e-6"

cmd = [
    obminimize_exe,
    "-ff", force_field,
    "-n", str(max_steps),
    "-c", convergence,
    str(input_ligand),
]

print("\nRunning obminimize...")
print("Command:")
print(" ".join(cmd))
print(f"Output file: {ligand_min_path.name}")

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Ligand minimization timed out after 600 seconds.\n"
        "   Check the ligand structure or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand minimization failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

# obminimize writes minimized structure to STDOUT
if not result.stdout.strip():
    raise ValueError(
        "\n obminimize did not return a minimized structure.\n"
        "   Inspect the input ligand before proceeding."
    )

with open(ligand_min_path, "w", encoding="utf-8") as fh:
    fh.write(result.stdout)

if not ligand_min_path.exists():
    raise FileNotFoundError(
        f"\n Minimized ligand file was not created:\n   {ligand_min_path}"
    )

# -------------------------------------------------------------------
# Basic validation
# -------------------------------------------------------------------
atom_count = 0
hydrogen_count = 0

with open(ligand_min_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element in ("H", "D"):
                hydrogen_count += 1

if atom_count == 0:
    raise ValueError(
        "\n The minimized ligand contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

print("\nLigand minimization summary")
print("-" * 60)
print(f"Input ligand:             {input_ligand.name}")
print(f"Minimized output:         {ligand_min_path.name}")
print(f"Force field:              {force_field}")
print(f"Maximum steps:            {max_steps}")
print(f"Convergence criterion:    {convergence}")
print(f"ATOM/HETATM records:      {atom_count}")
print(f"Hydrogen-like atoms:      {hydrogen_count}")

print("\n Note: MMFF94 is appropriate for typical organic ligands.")
print("   Ligands containing metals or unsupported elements may require")
print("   a different force field or a different preparation workflow.")

print("\n Visual inspection is recommended after minimization.")
print("   Compare the minimized ligand against the initial PDB in PyMOL")
print("   to confirm that the geometry was relaxed without distortions.")

# Optional variable for downstream steps
external_ligand_min_path = ligand_min_path

---

In [ ]:
# Step 5C3. Convert the minimized external ligand to PDBQT

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Input from Step 5C2
# -------------------------------------------------------------------
if "external_ligand_min_path" not in globals():
    raise NameError(
        "Variable 'external_ligand_min_path' was not found. Please run Step 5C2 first."
    )

input_ligand = Path(external_ligand_min_path)

if not input_ligand.exists():
    raise FileNotFoundError(
        f"\n Minimized ligand file not found:\n   {input_ligand}"
    )

print("Using minimized external ligand:")
print(f"  File: {input_ligand.name}")
print(f"  Path: {input_ligand.resolve()}")

# -------------------------------------------------------------------
# Detect Open Babel
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\n Open Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' or 'babel' is available."
    )

# -------------------------------------------------------------------
# Output file
# -------------------------------------------------------------------
ligand_pdbqt_path = input_ligand.with_suffix(".pdbqt")

if ligand_pdbqt_path.exists():
    ligand_pdbqt_path.unlink()



# -------------------------------------------------------------------
# Build command: PDB -> PDBQT
# Sin flags adicionales para permitir que Babel haga su trabajo por defecto:
# 1. Calcule cargas Gasteiger (necesarias para Vina)
# 2. Preserve todos los hidrógenos (All-atom)
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    "-ipdb", str(input_ligand),
    "-opdbqt",
    "-O", str(ligand_pdbqt_path),
    # Eliminamos -xn, -xp y -xh para no interferir con el cálculo de cargas
    # y la preservación de hidrógenos polares.
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Ligand PDBQT conversion timed out after 300 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand PDBQT conversion failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdbqt_path.exists():
    raise FileNotFoundError(
        f"\n Ligand PDBQT file was not created:\n   {ligand_pdbqt_path}"
    )

# -------------------------------------------------------------------
# Validation: compare atom counts
# -------------------------------------------------------------------
input_atoms = 0
with open(input_ligand, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            input_atoms += 1

output_atoms = 0
tree_records = 0
remark_lines = 0

with open(ligand_pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            output_atoms += 1
        elif line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            tree_records += 1
        elif line.startswith("REMARK"):
            remark_lines += 1

if output_atoms == 0:
    raise ValueError(
        "\n The generated ligand PDBQT contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

print("\nExternal ligand docking-format conversion summary")
print("-" * 60)
print(f"Input ligand:             {input_ligand.name}")
print(f"Output ligand PDBQT:      {ligand_pdbqt_path.name}")
print(f"Full path:                {ligand_pdbqt_path.resolve()}")
print(f"Input ATOM/HETATM:        {input_atoms}")
print(f"Output ATOM/HETATM:       {output_atoms}")
print(f"Tree-related records:     {tree_records}")
print(f"REMARK lines:             {remark_lines}")

if input_atoms == output_atoms:
    print("\n Atom count was preserved during conversion.")
else:
    print("\n Note: Atom count changed during conversion.")
    print(f"   Input atoms:  {input_atoms}")
    print(f"   Output atoms: {output_atoms}")
    print("   This is EXPECTED for AutoDock PDBQT format:")
    print("   Non-polar hydrogens (C-H) are represented implicitly")
    print("   by the atom types, not as explicit atoms.")
    print("   However, inspect the ligand carefully before docking.")

print("\n Visual inspection is recommended.")
print("   Open the ligand PDBQT in PyMOL or a text editor and confirm that:")
print("   - atom records are present")
print("   - the file is not empty or truncated")
print("   - the ligand identity is preserved after conversion")
print("   - the ligand is ready for docking with the prepared receptor")

# Optional variable for downstream steps
external_ligand_pdbqt_path = ligand_pdbqt_path

In [ ]:
# Revise the PDBQT file (Open Babel step)
from pathlib import Path

pdbqt_path = Path(
    r"C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_min.pdbqt")

atom_lines = []
hydrogen_lines = []
total_charge = 0.0

def extract_charge_from_pdbqt_line(line: str):
    """
    Extract partial charge from PDBQT.
    According to the PDBQT column definition:
    partialChrg = columns 71-76 -> Python slice [70:76]
    """
    if len(line) >= 76:
        charge_str = line[70:76].strip()
        try:
            return float(charge_str)
        except ValueError:
            pass

    # Fallback: whitespace-based parsing
    fields = line.split()
    if len(fields) >= 2:
        possible_charge = fields[-2]
        try:
            return float(possible_charge)
        except ValueError:
            pass

    return None

with open(pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            clean_line = line.rstrip()
            atom_lines.append(clean_line)

            atom_name = line[12:16].strip()

            # PDBQT atom type: columns 79-80 -> Python slice [78:80]
            atom_type = line[78:80].strip() if len(line) >= 80 else ""

            # Fallback if alignment is irregular
            if not atom_type:
                parts = line.split()
                if parts:
                    atom_type = parts[-1].strip()

            if atom_name.startswith("H") or atom_type in ("H", "HD", "D"):
                hydrogen_lines.append(clean_line)

            charge = extract_charge_from_pdbqt_line(line)
            if charge is not None:
                total_charge += charge

print(f"Total ATOM/HETATM lines: {len(atom_lines)}")
print(f"Explicit hydrogen atoms:    {len(hydrogen_lines)}")
print(f"TOTAL Charge calculated = {total_charge:.4f}")

print("\nFirst ATOM/HETATM records:")
for line in atom_lines[:20]:
    print(line)

if hydrogen_lines:
    print("\nExplicit hydrogen records (polar hydrogens only):")
    for line in hydrogen_lines:
        print(line)
else:
    print("\nNo explicit hydrogen records found (non-polar hydrogens are implicit).")

### Output Note – Steps 5C1 to 5C3

The external ligand was successfully processed from its original SDF format into a minimized, docking-ready PDBQT file using the Open Babel workflow.

In the first substep, the ligand structure was converted from SDF to PDB while hydrogens were added and protonation states were assigned for pH 7.4. This produced an intermediate PDB file containing the full all-atom representation of the molecule. In the second substep, the protonated structure underwent energy minimization using the MMFF94 force field to relax its local geometry. Finally, the minimized PDB was converted to PDBQT format, where the United-Atom model was applied to merge non‑polar hydrogens into their parent heavy atoms.

The resulting PDBQT file was saved in the `docking/sdf/` directory and contains the atom types, partial charges, and torsion‑tree information required for downstream docking with AutoDock Vina. At this stage, the external ligand is fully prepared and ready to be used together with the receptor in the docking setup.

In [ ]:
# Step 5D. Prepare an external ligand from SDF using Meeko
# Alternative workflow to compare against the Open Babel route.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# User-configurable variable
# If multiple SDF files exist in docking/sdf/, specify one explicitly.
# Example:
# user_ligand_sdf = "CID_10198397.sdf"
# -------------------------------------------------------------------
user_ligand_sdf = None

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "sdf").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\n Could not locate project root.\n"
        "   Make sure you are running this notebook from within the AutodockTutorial folder."
    )

sdf_dir = project_root / "docking" / "sdf"

if not sdf_dir.exists():
    raise FileNotFoundError(
        f"\n Required folder not found:\n   {sdf_dir}\n\n"
        "   Create the folder 'docking/sdf/' and place your ligand .sdf file there."
    )

# -------------------------------------------------------------------
# Select input SDF
# -------------------------------------------------------------------
if user_ligand_sdf is not None:
    user_ligand_sdf_path = Path(user_ligand_sdf)
    input_sdf = user_ligand_sdf_path if user_ligand_sdf_path.is_absolute() else sdf_dir / user_ligand_sdf_path.name
else:
    sdf_candidates = sorted(sdf_dir.glob("*.sdf"))

    if len(sdf_candidates) == 0:
        raise FileNotFoundError(
            f"\n No SDF ligand file was found in:\n   {sdf_dir}\n\n"
            "   Place your ligand .sdf file there, or set:\n"
            "   user_ligand_sdf = 'your_file.sdf'"
        )
    elif len(sdf_candidates) > 1:
        raise ValueError(
            "\n Multiple SDF files were found in docking/sdf/.\n"
            "   Please set the variable:\n"
            "   user_ligand_sdf = 'your_file.sdf'\n\n"
            f"   Detected files:\n   " + "\n   ".join(p.name for p in sdf_candidates)
        )
    else:
        input_sdf = sdf_candidates[0]

if not input_sdf.exists():
    raise FileNotFoundError(f"\nInput SDF file not found:\n   {input_sdf}")

if input_sdf.suffix.lower() != ".sdf":
    raise ValueError(f"\nExpected an SDF file, but received:\n   {input_sdf.name}")

# -------------------------------------------------------------------
# Important note about input chemistry
# -------------------------------------------------------------------
print("Using external ligand:")
print(f"  File: {input_sdf.name}")
print(f"  Path: {input_sdf.resolve()}")

print("\nNote:")
print("  This Meeko workflow assumes the input SDF already contains a valid 3D structure")
print("  and a chemically reasonable protonation state.")
print("  If needed, protonation and energy minimization should be handled before this step.")

# -------------------------------------------------------------------
# Detect Meeko tool
# -------------------------------------------------------------------
prepare_tool = (
    shutil.which("mk_prepare_ligand")
    or shutil.which("mk_prepare_ligand.py")
)

if prepare_tool is None:
    raise EnvironmentError(
        "\n Meeko ligand-preparation tool was not found.\n"
        "   Please install Meeko in the active environment.\n\n"
        "   Example:\n"
        "   pip install meeko\n\n"
        "   Then verify with:\n"
        "   where mk_prepare_ligand"
    )

# -------------------------------------------------------------------
# Define output
# -------------------------------------------------------------------
ligand_pdbqt_path = input_sdf.with_name(input_sdf.stem + "_meeko.pdbqt")

if ligand_pdbqt_path.exists():
    ligand_pdbqt_path.unlink()

# -------------------------------------------------------------------
# Build command
# Keep the command minimal and safe.
# Do NOT pass bare -p, because in Meeko that can be interpreted as
# load_atom_params / pdbqt-related option depending on the entry point.
# -------------------------------------------------------------------
cmd = [
    prepare_tool,
    "-i", str(input_sdf),
    "-o", str(ligand_pdbqt_path),
]

print("\nDetected preparation tool:")
print(f"  {prepare_tool}")

print("\nRunning Meeko ligand preparation...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Ligand preparation timed out after 300 seconds.\n"
        "   Check the ligand file or the Meeko installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand preparation from SDF failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdbqt_path.exists():
    raise FileNotFoundError(
        f"\n Ligand PDBQT file was not created:\n   {ligand_pdbqt_path}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDBQT
# PDBQT columns:
# partial charge ~ columns 71-76
# atom type ~ columns 79-80
# -------------------------------------------------------------------
atom_lines = []
hydrogen_lines = []
tree_records = 0
remark_lines = 0
total_charge = 0.0
charge_parse_errors = 0

with open(ligand_pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_lines.append(line.rstrip())

            atom_name = line[12:16].strip()
            atom_type = line[77:79].strip() if len(line) >= 79 else ""

            if atom_name.startswith("H") or atom_type in ("H", "HD", "HS"):
                hydrogen_lines.append(line.rstrip())

            try:
                partial_charge = float(line[70:76].strip())
                total_charge += partial_charge
            except ValueError:
                charge_parse_errors += 1

        elif line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            tree_records += 1
        elif line.startswith("REMARK"):
            remark_lines += 1

atom_count = len(atom_lines)
hydrogen_count = len(hydrogen_lines)

if atom_count == 0:
    raise ValueError(
        "\n The generated ligand PDBQT contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nMeeko external ligand preparation summary")
print("-" * 60)
print(f"Input SDF ligand:          {input_sdf.name}")
print(f"Preparation tool:          {Path(prepare_tool).name}")
print(f"Output ligand PDBQT:       {ligand_pdbqt_path.name}")
print(f"Full path:                 {ligand_pdbqt_path.resolve()}")
print(f"ATOM/HETATM records:       {atom_count}")
print(f"Explicit hydrogens:        {hydrogen_count}")
print(f"Tree-related records:      {tree_records}")
print(f"REMARK lines:              {remark_lines}")
print(f"Total charge calculated:   {total_charge:.4f}")

if charge_parse_errors > 0:
    print(f"Charge parse warnings:     {charge_parse_errors}")

if tree_records == 0:
    print("\n  Note: No tree records were detected.")
    print("   The ligand may have no rotatable bonds, or the output should be inspected manually.")

print("\nVisual inspection is recommended.")
print("  Open the ligand PDBQT in PyMOL or a text editor and confirm that:")
print("  - atom records are present")
print("  - the file is not empty or truncated")
print("  - the ligand identity is preserved")
print("  - explicit polar hydrogens are reasonable")
print("  - the ligand is ready for docking with the prepared receptor")

# Optional variable for downstream steps
external_ligand_meeko_pdbqt_path = ligand_pdbqt_path

In [ ]:
# Revise the PDBQT file (Meeko step)
from pathlib import Path

pdbqt_path = Path(
    r"C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_meeko.pdbqt")

atom_lines = []
hydrogen_lines = []
total_charge = 0.0

def extract_charge_from_pdbqt_line(line: str):
    """
    Extract partial charge from PDBQT.
    According to the PDBQT column definition:
    partialChrg = columns 71-76 -> Python slice [70:76]
    """
    if len(line) >= 76:
        charge_str = line[70:76].strip()
        try:
            return float(charge_str)
        except ValueError:
            pass

    # Fallback: whitespace-based parsing
    fields = line.split()
    if len(fields) >= 2:
        possible_charge = fields[-2]
        try:
            return float(possible_charge)
        except ValueError:
            pass

    return None

with open(pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            clean_line = line.rstrip()
            atom_lines.append(clean_line)

            atom_name = line[12:16].strip()

            # PDBQT atom type: columns 79-80 -> Python slice [78:80]
            atom_type = line[78:80].strip() if len(line) >= 80 else ""

            # Fallback if alignment is irregular
            if not atom_type:
                parts = line.split()
                if parts:
                    atom_type = parts[-1].strip()

            if atom_name.startswith("H") or atom_type in ("H", "HD", "D"):
                hydrogen_lines.append(clean_line)

            charge = extract_charge_from_pdbqt_line(line)
            if charge is not None:
                total_charge += charge

print(f"Total ATOM/HETATM lines: {len(atom_lines)}")
print(f"Explicit hydrogen atoms:    {len(hydrogen_lines)}")
print(f"TOTAL Charge calculated = {total_charge:.4f}")

print("\nFirst ATOM/HETATM records:")
for line in atom_lines[:20]:
    print(line)

if hydrogen_lines:
    print("\nExplicit hydrogen records (polar hydrogens only):")
    for line in hydrogen_lines:
        print(line)
else:
    print("\nNo explicit hydrogen records found (non-polar hydrogens are implicit).")

> **Output note:**  
> In this step, an **external ligand provided as an SDF file** was successfully prepared with **Meeko** as an alternative to the Open Babel workflow. The input file `CID_10198397.sdf` was converted directly into a docking-ready **PDBQT** file, `CID_10198397_meeko.pdbqt`, stored in the `docking/sdf/` directory.  
>
> The resulting file contained **17 ATOM/HETATM records**, **2 explicit hydrogen atoms**, and **8 tree-related records**, indicating that the ligand was written in a valid AutoDock-compatible representation with torsional information required for flexible docking. The reduced number of explicit hydrogens is consistent with the **PDBQT/AutoDock united-atom model**, in which non-polar hydrogens are typically not represented explicitly. Additionally, the total calculated charge was **-0.0010**, confirming a neutral ligand with properly assigned Gasteiger partial charges. 
>
> In practical terms, this step confirms that the external ligand could be prepared successfully with **Meeko**, generating a second valid docking-ready workflow that can now be compared against the corresponding **Open Babel** preparation route.